#ASSIGNMENT NLP – 3 (Chatbot using Transformers)


##Step 1 - Install

In [ ]:
!pip install transformers torch accelerate

##Step 2 - Import Libraries

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
print("✅ Step 2: Libraries imported!")

✅ Step 2: Libraries imported!


##Step 3 - Load DialoGPT-large

In [7]:
# Step 3: Load DialoGPT-large (COMPLETE FIX)
model_name = "microsoft/DialoGPT-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# ✅ PERFECT FIX for attention mask
tokenizer.pad_token = tokenizer.eos_token

print("✅ DialoGPT-large loaded! Warning fixed!")

def generate_response(prompt, chat_history_ids=None):
    """Fixed generation with attention mask"""
    new_user_input_ids = tokenizer.encode(prompt + tokenizer.eos_token, return_tensors='pt')

    bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1) if chat_history_ids is not None else new_user_input_ids

    # ✅ Attention mask fix
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    with torch.no_grad():
        outputs = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,  # ✅ FIX
            max_length=1000,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_p=0.9
        )

    response = tokenizer.decode(outputs[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
    return response, outputs

print("✅ Generation function ready - No warnings!")

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-large
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ DialoGPT-large loaded! Warning fixed!
✅ Generation function ready - No warnings!


##Step 4 - Chatbot Function

In [9]:
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")
    chat_history_ids = None

    while True:
        user_input = input("You: ")
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye!")
            break

        # FIXED Generation
        new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
        bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1) if chat_history_ids is not None else new_user_input_ids

        attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)  # ✅ FIX

        with torch.no_grad():
            chat_history_ids = model.generate(
                bot_input_ids,
                attention_mask=attention_mask,  # ✅ FIX
                max_length=1000,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
        print("Chatbot:", response)


In [10]:
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?

You: What is Artificial Intelligence?


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Chatbot: It's a type of AI.
You: Who created Python?
Chatbot: The AI created Python.
You: exit
Chatbot: Goodbye!
